# Learning Rate Sweep

Trains the same model over a grid of `(learning_rate, seed)` pairs.
Results are averaged across seeds so that conclusions about each LR are
seed-independent.

## Imports

In [1]:
import json
import itertools
from pathlib import Path

import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from qvarnet.train import train
from qvarnet.models.deep_set import DeepSet
from qvarnet.hamiltonian.continuous import HarmonicOscillatorHamiltonian

## System

In [2]:
N_PARTICLES = 50
DIM         = 1
N_CHAINS    = 5_000
DoF         = N_PARTICLES * DIM
SHAPE       = (N_CHAINS, DoF)

## Model Factory

A function is used instead of a single instance so that each `(lr, seed)` run
starts from an independently initialised model.

In [3]:
IS_LOG_MODEL = True

def make_model():
    return DeepSet(
        phi_hidden_architecture=[2],
        F_hidden_architecture=[1],
        hidden_internal_dimension=1,
        n_particles=N_PARTICLES,
    )

## Hamiltonian

In [4]:
OMEGA = 1.0
E_EXACT = 0.5 * OMEGA  # ground state: 0.5 ℏω (ℏ = 1)

hamiltonian = HarmonicOscillatorHamiltonian(omega=OMEGA)

## Sampler

In [5]:
STEP_SIZE            = 0.5
CHAIN_LENGTH         = 11
THERMALIZATION_STEPS = 10
THINNING_FACTOR      = 1
PBC                  = 40.0

sampler_params = {
    "step_size":            STEP_SIZE,
    "chain_length":         CHAIN_LENGTH + 1,
    "thermalization_steps": THERMALIZATION_STEPS,
    "thinning_factor":      THINNING_FACTOR,
    "PBC":                  PBC,
}

## Training Hyperparameters

In [6]:
N_EPOCHS            = 10_000
WARM_WALKERS        = True
IS_UPDATE_STEP_SIZE = True
MIN_STEP            = 1e-5
MAX_STEP            = 5.0
SAVE_CHECKPOINTS    = False
CHECKPOINT_PATH     = "./"

## Sweep Configuration

Edit `LR_LIST` and `SEEDS` to define the grid.

In [7]:
# LR_LIST = [1e-3, 5e-3, 1e-2, 2.5e-2, 5e-2]
# SEEDS   = [0, 1, 2, 3, 4]
# LR_LIST = [0.025, 0.05, 0.04, 0.03, 0.02, 0.005, 0.001]
SEEDS  = [0, 1]
LR_LIST = [0.05,0.025,0.005,0.001]

print(f"{len(LR_LIST)} LRs × {len(SEEDS)} seeds = {len(LR_LIST)*len(SEEDS)} runs")
print(f"LRs  : {LR_LIST}")
print(f"Seeds: {SEEDS}")

4 LRs × 2 seeds = 8 runs
LRs  : [0.05, 0.025, 0.005, 0.001]
Seeds: [0, 1]


## Training Loop

All `(lr, seed)` combinations are run sequentially.
Raw histories are stored in `raw_results[lr][seed]`.

In [ ]:
# raw_results[lr][seed] = dict of 1-D numpy arrays (one value per epoch)
raw_results = {lr: {} for lr in LR_LIST}

total = len(LR_LIST) * len(SEEDS)
done  = 0

for lr, seed in itertools.product(LR_LIST, SEEDS):
    done += 1
    print(f"[{done}/{total}]  lr={lr:.0e}  seed={seed}", flush=True)

    history, _, _ = train(
        n_epochs=N_EPOCHS,
        shape=SHAPE,
        model=make_model(),
        optimizer=optax.adam(learning_rate=lr),
        sampler_params=sampler_params,
        hamiltonian=hamiltonian,
        rng_seed=seed,
        warm_walkers=WARM_WALKERS,
        is_update_step_size=IS_UPDATE_STEP_SIZE,
        is_log_model=IS_LOG_MODEL,
        min_step=MIN_STEP,
        max_step=MAX_STEP,
        save_checkpoints=SAVE_CHECKPOINTS,
        checkpoint_path=CHECKPOINT_PATH,
    )

    raw_results[lr][seed] = {
        "energies"  : np.array([s.energy for s in history]) / N_PARTICLES,
        "stds"      : np.array([s.std    for s in history]) / N_PARTICLES,
        "acc_rates" : np.array([float(jnp.mean(s.acceptance_rate)) for s in history]),
        "step_sizes": np.array([float(s.step_size) for s in history]),
    }

print("Done.")

[1/8]  lr=5e-02  seed=0


100%|██████████| 10000/10000 [02:41<00:00, 61.89it/s, E=29.38, sigma_E=4.83]


[2/8]  lr=5e-02  seed=1


100%|██████████| 10000/10000 [02:41<00:00, 61.74it/s, E=29.26, sigma_E=4.52]


[3/8]  lr=3e-02  seed=0


...

100%|██████████| 10000/10000 [02:40<00:00, 62.43it/s, E=25.01, sigma_E=0.22]


[6/8]  lr=5e-03  seed=1


 68%|██████▊   | 6797/10000 [01:49<00:51, 62.00it/s, E=25.04, sigma_E=0.44]


KeyboardInterrupt: 

: 

# When things go wrong

In [ ]:
def smooth(x, window):
    """Uniform moving average.  Edges are filled with the raw value."""
    if window <= 1:
        return x.copy()
    kernel = np.ones(window) / window
    out = np.convolve(x, kernel, mode="same")
    # fix edge artefacts from zero-padding
    half = window // 2
    for i in range(half):
        out[i]      = x[: i + half + 1].mean()
        out[-i - 1] = x[-(i + half + 1) :].mean()
    return out

def detect_plateau_running_min(
    energies,
    smooth_window=50,
    rel_tol=0.02,
    patience=200,
):
    """
    Parameters
    ----------
    energies     : 1-D array of raw energies.
    smooth_window: width of the moving-average smoother.
    rel_tol      : fractional threshold above running min to count as "worse".
                   E.g. 0.02 → 2 % above best seen so far.
    patience     : consecutive epochs above threshold before flagging.

    Returns
    -------
    mask         : bool array, True where plateau is active.
    first_epoch  : int, first epoch flagged (-1 if never).
    running_min  : smoothed running-minimum array.
    smoothed     : smoothed energy array.
    """
    smoothed    = smooth(energies, smooth_window)
    running_min = np.minimum.accumulate(smoothed)

    above   = smoothed > running_min * (1.0 + rel_tol)  # worse than best by rel_tol
    mask    = np.zeros(len(energies), dtype=bool)
    counter = 0
    flagging = False

    for i, a in enumerate(above):
        if a:
            counter += 1
        else:
            counter  = 0
            flagging = False
        if counter >= patience:
            flagging = True
        if flagging:
            mask[i] = True

    first_epoch = int(np.argmax(mask)) if mask.any() else -1
    return mask, first_epoch, running_min, smoothed


# ── Tune these ───────────────────────────────────────────────────────────────
SMOOTH_W  = 50
REL_TOL   = 0.02   # 2 % above running min
PATIENCE  = 200    # epochs
# ─────────────────────────────────────────────────────────────────────────────

energies = np.array([s.energy for s in raw_results[LR_LIST[0]][SEEDS[0]]["energies"]])
epochs   = np.arange(len(energies))

mask1, first1, running_min1, smoothed1 = detect_plateau_running_min(
    energies, smooth_window=SMOOTH_W, rel_tol=REL_TOL, patience=PATIENCE
)

print(f"Detector 1 — Running-min + patience")
print(f"  First plateau epoch : {first1}")
print(f"  Plateau fraction    : {mask1.mean():.1%}")

## Aggregate over Seeds

For each LR compute mean and std across seeds, epoch by epoch.

In [ ]:
# agg[lr] = dict of 1-D arrays with shape (N_EPOCHS,)
agg = {}

for lr in LR_LIST:
    seed_energies   = np.stack([raw_results[lr][s]["energies"]   for s in SEEDS])  # (n_seeds, n_epochs)
    seed_stds       = np.stack([raw_results[lr][s]["stds"]       for s in SEEDS])
    seed_acc_rates  = np.stack([raw_results[lr][s]["acc_rates"]  for s in SEEDS])
    seed_step_sizes = np.stack([raw_results[lr][s]["step_sizes"] for s in SEEDS])

    agg[lr] = {
        "energy_mean"   : seed_energies.mean(axis=0),
        "energy_std"    : seed_energies.std(axis=0),
        "sigma_e_mean"  : seed_stds.mean(axis=0),
        "acc_rate_mean" : seed_acc_rates.mean(axis=0),
        "step_size_mean": seed_step_sizes.mean(axis=0),
    }

epochs = np.arange(N_EPOCHS)
print("Aggregation done.")

## Save Results

Both the per-seed raw data and the seed-aggregated stats are written to ASCII
JSON so they can be reloaded without re-running the sweep.
Compress afterwards with `gzip -k results_raw.json`.

In [ ]:
def _to_serialisable(d):
    """Recursively convert numpy arrays / float keys to JSON-safe types."""
    if isinstance(d, dict):
        return {str(k): _to_serialisable(v) for k, v in d.items()}
    if isinstance(d, np.ndarray):
        return d.tolist()
    return d


def save_sweep(raw_results, agg, path_raw="sweep_raw.json", path_agg="sweep_agg.json"):
    meta = {"lr_list": LR_LIST, "seeds": SEEDS, "n_epochs": N_EPOCHS, "e_exact": E_EXACT}

    with open(path_raw, "w") as f:
        json.dump({"meta": meta, "data": _to_serialisable(raw_results)}, f, separators=(",", ":"))

    with open(path_agg, "w") as f:
        json.dump({"meta": meta, "data": _to_serialisable(agg)}, f, separators=(",", ":"))

    for p in (path_raw, path_agg):
        print(f"Saved {p}  ({Path(p).stat().st_size/1024:.0f} KB)")


save_sweep(raw_results, agg)

## Load Results

Run this cell (and skip the training loop above) to restore a previous sweep.

In [ ]:
def load_sweep(path_raw="sweep_raw.json", path_agg="sweep_agg.json"):
    def _from_serialisable(d):
        if isinstance(d, dict):
            return {k: _from_serialisable(v) for k, v in d.items()}
        if isinstance(d, list):
            arr = np.array(d)
            return arr if arr.dtype != object else d
        return d

    with open(path_raw) as f:
        raw_data = json.load(f)
    with open(path_agg) as f:
        agg_data = json.load(f)

    meta = raw_data["meta"]

    # restore float keys for lr
    raw = {float(lr): {int(s): _from_serialisable(v)
                       for s, v in seeds.items()}
           for lr, seeds in raw_data["data"].items()}
    agg = {float(lr): _from_serialisable(v)
           for lr, v in agg_data["data"].items()}

    print(f"Loaded: {len(raw)} LRs, {len(next(iter(raw.values())))} seeds, "
          f"{meta['n_epochs']} epochs each")
    return raw, agg, meta


# Uncomment to reload without re-training:
# raw_results, agg, _meta = load_sweep()
# LR_LIST = _meta["lr_list"]; SEEDS = _meta["seeds"]
# N_EPOCHS = _meta["n_epochs"]; E_EXACT = _meta["e_exact"]
# epochs = np.arange(N_EPOCHS)

## Plot — Energy Convergence per LR

Each LR gets one line (mean over seeds) with a ±std band.

In [ ]:
colors = cm.viridis(np.linspace(0.1, 0.9, len(LR_LIST)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for color, lr in zip(colors, LR_LIST):
    mu  = agg[lr]["energy_mean"]
    sig = agg[lr]["energy_std"]
    label = f"lr={lr:.0e}"

    axes[0].plot(epochs, mu, lw=0.9, color=color, label=label)
    axes[0].fill_between(epochs, mu - sig, mu + sig, alpha=0.15, color=color)

    err = np.abs(mu - E_EXACT)
    axes[1].semilogy(epochs, err, lw=0.9, color=color, label=label)

for ax in axes:
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlabel("Epoch")

axes[0].axhline(E_EXACT, color="red", ls="--", lw=0.8, label=f"E₀={E_EXACT}")
axes[0].set_ylabel("Energy / particle")
axes[0].set_title("Energy convergence (mean ± std over seeds)")

axes[1].set_ylabel("|E − E₀|")
axes[1].set_title("Absolute error (log scale)")

plt.tight_layout()
plt.savefig("sweep_energy.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot — Final Energy Comparison (Bar Chart)

Mean final energy ± std over seeds for each LR (last `TAIL` epochs).

In [ ]:
TAIL = 100

final_means = [agg[lr]["energy_mean"][-TAIL:].mean() for lr in LR_LIST]
final_stds  = [agg[lr]["energy_std" ][-TAIL:].mean() for lr in LR_LIST]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(LR_LIST))
bars = ax.bar(x, final_means, yerr=final_stds, capsize=4,
              color=colors, alpha=0.85, edgecolor="k", linewidth=0.6)
ax.axhline(E_EXACT, color="red", ls="--", lw=0.9, label=f"E₀ = {E_EXACT}")
ax.set_xticks(x)
ax.set_xticklabels([f"{lr:.0e}" for lr in LR_LIST], rotation=30, ha="right")
ax.set_xlabel("Learning rate")
ax.set_ylabel("Energy / particle")
ax.set_title(f"Final energy (mean over last {TAIL} epochs and {len(SEEDS)} seeds)")
ax.legend()
plt.tight_layout()
plt.savefig("sweep_bar.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'LR':>10}  {'mean E':>10}  {'std E':>10}  {'|E-E₀|':>10}")
for lr, m, s in zip(LR_LIST, final_means, final_stds):
    print(f"{lr:>10.0e}  {m:>10.6f}  {s:>10.6f}  {abs(m-E_EXACT):>10.6f}")

## Plot — Acceptance Rate and Step Size per LR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for color, lr in zip(colors, LR_LIST):
    label = f"lr={lr:.0e}"
    axes[0].plot(epochs, agg[lr]["acc_rate_mean"],  lw=0.9, color=color, label=label)
    axes[1].plot(epochs, agg[lr]["step_size_mean"], lw=0.9, color=color, label=label)

axes[0].axhline(0.5, color="red", ls="--", lw=0.8, label="target 0.5")
axes[0].set_ylabel("Acceptance rate"); axes[0].set_xlabel("Epoch")
axes[0].set_title("MH acceptance rate (mean over seeds)")
axes[0].legend(fontsize=8)

axes[1].set_ylabel("Step size"); axes[1].set_xlabel("Epoch")
axes[1].set_title("MH step size (mean over seeds)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("sweep_sampler.png", dpi=150, bbox_inches="tight")
plt.show()

## Per-Seed Raw Curves

Pick a single LR and overlay all individual seed runs to see variability.

In [ ]:
INSPECT_LR = LR_LIST[2]  # change index to inspect a different LR

fig, ax = plt.subplots(figsize=(8, 4))
seed_colors = cm.plasma(np.linspace(0.15, 0.85, len(SEEDS)))

for sc, seed in zip(seed_colors, SEEDS):
    ax.plot(epochs, raw_results[INSPECT_LR][seed]["energies"],
            lw=0.7, alpha=0.7, color=sc, label=f"seed {seed}")

ax.plot(epochs, agg[INSPECT_LR]["energy_mean"],
        lw=1.5, color="black", ls="--", label="mean")
ax.axhline(E_EXACT, color="red", ls=":", lw=0.9, label=f"E₀={E_EXACT}")
ax.set_xlabel("Epoch"); ax.set_ylabel("Energy / particle")
ax.set_title(f"All seeds for lr={INSPECT_LR:.0e}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Param / Gradient Dashboard for Best LR

Runs one extra training pass for the best-performing LR and seed so that the
full `state_history` (params + grads per epoch) is available for the dashboard.
Skip if you do not need per-parameter diagnostics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from plot_param_dashboard import compute_stats, save_stats, load_stats, plot_dashboard

# Pick the LR with the lowest mean final error
errors      = {lr: abs(agg[lr]["energy_mean"][-TAIL:].mean() - E_EXACT) for lr in LR_LIST}
BEST_LR     = min(errors, key=errors.get)
BEST_SEED   = SEEDS[0]
print(f"Best LR: {BEST_LR:.0e}  (|E-E₀| = {errors[BEST_LR]:.6f})")

print("Re-running for param/grad history…")
best_history, _, _ = train(
    n_epochs=N_EPOCHS,
    shape=SHAPE,
    model=make_model(),
    optimizer=optax.adam(learning_rate=BEST_LR),
    sampler_params=sampler_params,
    hamiltonian=hamiltonian,
    rng_seed=BEST_SEED,
    warm_walkers=WARM_WALKERS,
    is_update_step_size=IS_UPDATE_STEP_SIZE,
    is_log_model=IS_LOG_MODEL,
    min_step=MIN_STEP,
    max_step=MAX_STEP,
    save_checkpoints=False,
    checkpoint_path=CHECKPOINT_PATH,
)

In [ ]:
param_stats = compute_stats(best_history, target="params")
grad_stats  = compute_stats(best_history, target="grads")

save_stats(param_stats, f"best_lr{BEST_LR:.0e}_param_stats.json")
save_stats(grad_stats,  f"best_lr{BEST_LR:.0e}_grad_stats.json")

In [ ]:
plot_dashboard(param_stats, mode="mean_std",
               title=f"Params — mean ± std  (lr={BEST_LR:.0e})")

In [ ]:
plot_dashboard(param_stats, mode="norm",
               title=f"Params — L2 norm  (lr={BEST_LR:.0e})")

In [ ]:
plot_dashboard(grad_stats, mode="mean_std",
               title=f"Grads — mean ± std  (lr={BEST_LR:.0e})")

In [ ]:
plot_dashboard(grad_stats, mode="norm", log_scale=True,
               title=f"Grads — L2 norm  (lr={BEST_LR:.0e})")